In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

2026-05-04 23:33:07.531751: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/franceliatracygaxiola/Downloads/AI-Course-main/03 - Code/Module 2/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

 1. Load Data with Augmentation

In [6]:
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory('./train', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='sparse')
val_gen = val_datagen.flow_from_directory('./data/val', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='sparse')

Found 93 images belonging to 5 classes.


FileNotFoundError: [Errno 2] No such file or directory: './data/val'

In [ ]:
train_gen.class_indices.keys()

dict_keys(['ben_afflek', 'elton_john', 'jerry_seinfeld', 'madonna', 'mindy_kaling'])

2. Build Model (Transfer Learning)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False 

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(len(train_gen.class_indices), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

3. Train and Save

In [ ]:
model.fit(train_gen, validation_data=val_gen, epochs=10) # Cambié a 10 epocas para mejorar la precision
model.save('celebrity_model.h5')

/Users/franceliatracygaxiola/Downloads/AI-Course-main/03 - Code/Module 2/.venv/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 902ms/step - accuracy: 0.2623 - loss: 1.7459 - val_accuracy: 0.2800 - val_loss: 1.5539
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 377ms/step - accuracy: 0.5123 - loss: 1.2620 - val_accuracy: 0.4000 - val_loss: 1.4495
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 411ms/step - accuracy: 0.5863 - loss: 1.2001 - val_accuracy: 0.4800 - val_loss: 1.2603
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 378ms/step - accuracy: 0.6856 - loss: 0.9141 - val_accuracy: 0.5200 - val_loss: 1.0801
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 359ms/step - accuracy: 0.7181 - loss: 0.7982 - val_accuracy: 0.6800 - val_loss: 0.9570
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 398ms/step - accuracy: 0.7427 - loss: 0.6701 - val_accuracy: 0.8000 - val_loss: 0.8661
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 416ms/step - accuracy: 0.7140 - loss: 0.6995 - val_accuracy: 0.8000 - val_loss: 0.8017
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 352ms/step - accuracy: 0.8513 - loss: 0.5389 - val_accuracy: 0.7600 - val_loss:

Store class names for prediction

In [ ]:
class_names = list(train_gen.class_indices.keys())
print("Classes identified:", class_names)

Classes identified: ['ben_afflek', 'elton_john', 'jerry_seinfeld', 'madonna', 'mindy_kaling']


Testing

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

def predict_celebrity(img_path, model_path='celebrity_model.h5'):
    # Load model and image
    model = tf.keras.models.load_model(model_path)
    img = image.load_img(img_path, target_size=(160, 160))
    
    # Pre-process
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0) # Create batch axis

    # Predict
    predictions = model.predict(img_array)
    score = np.max(predictions)
    class_idx = np.argmax(predictions)
    
    # Display
    plt.imshow(img)
    plt.title(f"Prediction: {class_names[class_idx]} ({100 * score:.2f}%)")
    plt.axis('off')
    plt.show()

# Usage:
predict_celebrity('Ben-Affleck.webp')
predict_celebrity('Madonna.webp')
predict_celebrity('Elton-John.webp')
predict_celebrity('Jerry-Seinfeld.webp')
predict_celebrity('Mindy-Kaling.webp')

ValueError: File not found: filepath=../celebrity_model.keras. Please ensure the file is an accessible `.keras` zip file.